# **ETL of World Bank Temperature Data**

## Objectives
* read in World Bank's Temperature data
* prepare the data
* append to CO2 and population and GDP data

## Inputs
* data/processed/01d_df_co2_pop_gdp_1990_2024.csv
* World Bank's Global Temperature data via an api

## Outputs

* data/processed/01e_df_co2_pop_gdp_degc_1990_2024.csv


---

# Set Up directories and Import Necessary Libraries

In [12]:
# System and OS related tasks
import sys
import os

# Add the project root to Python path
project_root = os.path.abspath('..')
sys.path.insert(0, project_root)

# path to directories
raw_dir = '../data/raw'
processed_dir = '../data/processed'

In [13]:
# Get the necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid') # sets a white background with grid lines 

## 1.0 Get the List of Countries from the CO2-Population-GDP data

In [14]:
data_columns = [
    "country_code_iso3"
    , "country_name_iso3"
    , "year"
    , "co2_pc"
    , "population"
    , "gdp_constant_ppp_pc"
]

column_dtypes = {
    "country_code_iso3": str
    , "country_name_iso3": str
    , "year": int
    , "co2_pc" : float
    , "population" : float
    , "gdp_constant_ppp_pc": float
    }

In [15]:
file_name = "01d_df_co2_pop_gdp_1990_2024.csv"

df_co2_pop_gdp_1990_2024 = pd.read_csv(
    f"{processed_dir}/{file_name}",
    header=0,
    names=data_columns,
    dtype=column_dtypes
    )

print(f"Total number of rows: {len(df_co2_pop_gdp_1990_2024)}")

Total number of rows: 7473


In [16]:
df_co2_pop_gdp_1990_2024.head(3)

,country_code_iso3,country_name_iso3,year,co2_pc,population,gdp_constant_ppp_pc
0,AFG,Afghanistan,1990,0.168054,12045664.0,NaN
1,AFG,Afghanistan,1991,0.156411,12238879.0,NaN
2,AFG,Afghanistan,1992,0.111609,13278983.0,NaN


In [9]:
country_list = sorted(list(set(df_co2_pop_gdp_1990_2024["country_code_iso3"])))
country_list

['ABW',
 'AFG',
 'AGO',
 'AIA',
 'ALB',
 'AND',
 'ARE',
 'ARG',
 'ARM',
 'ATG',
 'AUS',
 'AUT',
 'AZE',
 'BDI',
 'BEL',
 'BEN',
 'BES',
 'BFA',
 'BGD',
 'BGR',
 'BHR',
 'BHS',
 'BIH',
 'BLR',
 'BLZ',
 'BMU',
 'BOL',
 'BRA',
 'BRB',
 'BRN',
 'BTN',
 'BWA',
 'CAF',
 'CAN',
 'CHE',
 'CHL',
 'CHN',
 'CIV',
 'CMR',
 'COD',
 'COG',
 'COK',
 'COL',
 'COM',
 'CPV',
 'CRI',
 'CUB',
 'CUW',
 'CYP',
 'CZE',
 'DEU',
 'DJI',
 'DMA',
 'DNK',
 'DOM',
 'DZA',
 'ECU',
 'EGY',
 'ERI',
 'ESP',
 'EST',
 'ETH',
 'FIN',
 'FJI',
 'FRA',
 'FRO',
 'FSM',
 'GAB',
 'GBR',
 'GEO',
 'GHA',
 'GIN',
 'GMB',
 'GNB',
 'GNQ',
 'GRC',
 'GRD',
 'GRL',
 'GTM',
 'GUY',
 'HKG',
 'HND',
 'HRV',
 'HTI',
 'HUN',
 'IDN',
 'IND',
 'IRL',
 'IRN',
 'IRQ',
 'ISL',
 'ISR',
 'ITA',
 'JAM',
 'JOR',
 'JPN',
 'KAZ',
 'KEN',
 'KGZ',
 'KHM',
 'KIR',
 'KNA',
 'KOR',
 'KWT',
 'LAO',
 'LBN',
 'LBR',
 'LBY',
 'LCA',
 'LIE',
 'LKA',
 'LSO',
 'LTU',
 'LUX',
 'LVA',
 'MAC',
 'MAR',
 'MDA',
 'MDG',
 'MDV',
 'MEX',
 'MHL',
 'MKD',
 'MLI',
 'MLT',


In [10]:
print(f"Total number of countries: {len(country_list)}")

Total number of countries: 214


## 2.0 World Bank's Temperature Data using the Climate Change Knowledge Portal (CCKP)

📌 The following parameters were used to generate temperature data for all countries:
* Area of Focus: Global Countries (global_countries)
* Collection: CRU 0.5-degree (cru-x0.5)
  * The World Bank's CCKP uses the 0.5-degree resolution CRU dataset because it represents a unique combination of long-term historical coverage, reliability for trend analysis, and suitability for national-scale assessments
* Type: Timeseries (timeseries)
* Variable: Average Mean Surface Air Temperature (`tas`)
* Product: Time Series (timeseries)
* Aggregation: Annual (annual)
* Period: 1901-2024

In [17]:
import requests

url_temp = "https://cckpapi.worldbank.org/api/v1/cru-x0.5_timeseries_tas_timeseries_annual_1901-2024_mean_historical_cru_ts4.09_mean/global_countries?_format=json"

response_temp = requests.get(url_temp)
wb_temp_all_dict = response_temp.json() 

📌 Taking a look at the returned JSON:

In [18]:
# take a look at the wb_temp_all_dict JSON
print("Type:", type(wb_temp_all_dict)) 
print("Top-level keys:", wb_temp_all_dict.keys())
print("Number of top-level items:", len(wb_temp_all_dict))

Type: <class 'dict'>
Top-level keys: dict_keys(['metadata', 'data'])
Number of top-level items: 2


📌 It looks like there are 2 dictionaries inside with keys `metadata` and `data`.  Take peek into both of them:

In [19]:
# take a look at the metadata and data
print("Metadata:", wb_temp_all_dict.get('metadata', {}))
print("Data:", wb_temp_all_dict.get('data', {}))

Metadata: {'apiVersion': 'v1', 'status': 'success', 'messages': []}
Data: {'ABW': {'1901-07': 28.22, '1902-07': 27.79, '1903-07': 27.89, '1904-07': 27.62, '1905-07': 27.68, '1906-07': 27.58, '1907-07': 27.56, '1908-07': 27.46, '1909-07': 27.52, '1910-07': 27.52, '1911-07': 27.59, '1912-07': 27.65, '1913-07': 27.48, '1914-07': 27.57, '1915-07': 28.35, '1916-07': 28.1, '1917-07': 27.8, '1918-07': 27.82, '1919-07': 27.63, '1920-07': 27.5, '1921-07': 27.65, '1922-07': 27.52, '1923-07': 27.94, '1924-07': 28.1, '1925-07': 27.82, '1926-07': 28.35, '1927-07': 27.84, '1928-07': 28, '1929-07': 27.82, '1930-07': 27.98, '1931-07': 28.31, '1932-07': 28.51, '1933-07': 28.37, '1934-07': 28.49, '1935-07': 28.18, '1936-07': 28.41, '1937-07': 28.34, '1938-07': 28.04, '1939-07': 28.29, '1940-07': 28.84, '1941-07': 29.02, '1942-07': 28.74, '1943-07': 28.2, '1944-07': 28.12, '1945-07': 27.94, '1946-07': 28.07, '1947-07': 28.65, '1948-07': 28.28, '1949-07': 28.03, '1950-07': 27.57, '1951-07': 28.24, '1952-0

📌 It looks like the temperature data is stored inside the data dictionary of the JSON with country code as the key. 

📌 The temperature data actually goes back to year 1901 but we only need data for years 1990-2024.

📌 Now get the data dictionary out of the JSON stored in wb_temp_all_dict.

In [20]:
wb_temp_country_dict = wb_temp_all_dict['data']

print(f"Number of key-value pairs inside wb_temp_country_dict: {len(wb_temp_country_dict)}")

Number of key-value pairs inside wb_temp_country_dict: 246


📌 We only want weather temperature data for the 214 countries.

In [37]:
country_temp_dict = {}

for country in country_list:
    if country in wb_temp_country_dict:
        country_temp_dict[country] = wb_temp_country_dict[country]
              
print(f"Number of records: {len(country_temp_dict)}")

Number of records: 213


📌 There is 1 missing country and that is Kosovo!

In [23]:
# Find countries in your list that are NOT in the temperature data
missing_countries = set(country_list) - set(wb_temp_country_dict.keys())

missing_countries

{'XKX'}

📌 Convert the dictionary into a dataframe:
* clean the year data which is the form of "1901-07" into a 4 digit year and 
* then put the country, year, temp data into long form dataframe.

In [ ]:
country_temp_list = []

# loop through to get the country and the {year: temp} out
for country, year_dict in country_temp_dict.items():
    # loop through to get the year and temperature out
    for year_month, degc in year_dict.items():
        # get the year from "1901-07" format by splitting by "-" and take the left side where index = 0
        year = int(year_month.split("-")[0])

        # add to the list for data only for period 1990 to 2024
        if year >= 1990 and year <=2024:
            country_temp_list.append({
                "country_code_iso3":  country
                , "year": year
                , "degc": degc
            })

# convert country_temp_list  into a dataframe
df_country_temp_1990_2024 = pd.DataFrame(country_temp_list)        

df_country_temp_1990_2024.head(5)


,country_code_iso3,year,degc
0,ABW,1990,28.07
1,ABW,1991,28.66
2,ABW,1992,28.94
3,ABW,1993,28.95
4,ABW,1994,29.02


📌 Check if each country has temp data for each of the years 2000-2024.

In [32]:
country_temp_stats = df_country_temp_1990_2024.groupby("country_code_iso3").agg(
    {"year": ["min", "max","count","nunique"]}
)

country_temp_stats

year                    
                    min   max count nunique
country_code_iso3                          
ABW                1990  2024    35      35
AFG                1990  2024    35      35
AGO                1990  2024    35      35
AIA                1990  2024    35      35
ALB                1990  2024    35      35
...                 ...   ...   ...     ...
WSM                1990  2024    35      35
YEM                1990  2024    35      35
ZAF                1990  2024    35      35
ZMB                1990  2024    35      35
ZWE                1990  2024    35      35

[213 rows x 4 columns]

---

# 3.0 Merge Temperature data with CO2, Population and GDP Data

📌 Get the CO2, population and CO2 data.

In [26]:
df_co2_pop_gdp_1990_2024.head(5)

,country_code_iso3,country_name_iso3,year,co2_pc,population,gdp_constant_ppp_pc
0,AFG,Afghanistan,1990,0.168054,12045664.0,NaN
1,AFG,Afghanistan,1991,0.156411,12238879.0,NaN
2,AFG,Afghanistan,1992,0.111609,13278983.0,NaN
3,AFG,Afghanistan,1993,0.099506,14943175.0,NaN
4,AFG,Afghanistan,1994,0.089462,16250800.0,NaN


📌 Merge the CO2, population and CO2 data with temperature data:

In [33]:
df_co2_pop_gdp_degc_1990_2024 = df_co2_pop_gdp_1990_2024.merge(
    df_country_temp_1990_2024[["country_code_iso3", "year", "degc"]]
    , on=["country_code_iso3", "year"]
    , how="left"
)

df_co2_pop_gdp_degc_1990_2024.head(5)

,country_code_iso3,country_name_iso3,year,co2_pc,population,gdp_constant_ppp_pc,degc
0,AFG,Afghanistan,1990,0.168054,12045664.0,NaN,13.25
1,AFG,Afghanistan,1991,0.156411,12238879.0,NaN,12.70
2,AFG,Afghanistan,1992,0.111609,13278983.0,NaN,12.42
3,AFG,Afghanistan,1993,0.099506,14943175.0,NaN,12.71
4,AFG,Afghanistan,1994,0.089462,16250800.0,NaN,12.95


In [38]:
print(f"Total number of rows in df_co2_pop_gdp_degc_1990_2024: {len(df_co2_pop_gdp_degc_1990_2024)}")
print(f"Rows with temperature data: {df_co2_pop_gdp_degc_1990_2024['degc'].notna().sum()}")
print(f"Rows missing temperature: {df_co2_pop_gdp_degc_1990_2024['degc'].isna().sum()}")

Total number of rows in df_co2_pop_gdp_degc_1990_2024: 7473
Rows with temperature data: 7442
Rows missing temperature: 31


📌 Check the 31 rows with missing temperature and the country is Kosovo.

In [ ]:
missing_gdp_rows = df_co2_pop_gdp_degc_1990_2024[df_co2_pop_gdp_degc_1990_2024["degc"].isna()].copy()

missing_gdp_rows["country_name_iso3"].value_counts()

country_name_iso3
Kosovo    31
Name: count, dtype: int64

---

# 4.0 Export to CSV

In [39]:
df_co2_pop_gdp_degc_1990_2024.to_csv(f'{processed_dir}/01e_df_co2_pop_gdp_degc_1990_2024.csv', index=False)
print(f"Exported: {processed_dir}/01e_df_co2_pop_gdp_degc_1990_2024.csv")

Exported: ../data/processed/01e_df_co2_pop_gdp_degc_1990_2024.csv


---